In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path

# Configuration
REPO_URL = (
    "https://github.com/"
    "bisalzindal-blip/"
    "Prediction-of-the-Optimal-NaCl-Concentration-for-Bacterial-Growth.git"
)

CLONE_DIR = Path(
    "/content/Prediction-of-the-Optimal-NaCl-Concentration-for-Bacterial-Growth"
)

FINAL_TEST_MAE = 1.3354  # Reference test-set MAE

print("=" * 70)
print("BACion")
print("Prediction of the Maximum NaCl Concentration Supporting Bacterial Growth")
print("=" * 70)

os.chdir("/content")

if CLONE_DIR.exists():
    shutil.rmtree(CLONE_DIR)

clone = subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)],
    cwd="/content",
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

if clone.returncode != 0:
    raise RuntimeError(
        f"Failed to clone BACion repository:\n{clone.stderr}"
    )

possible_dirs = [
    CLONE_DIR,
    CLONE_DIR / "BACion"
]

BACON_DIR = None

for directory in possible_dirs:
    if (
        (directory / "pyproject.toml").exists()
        and (directory / "src" / "bacion").exists()
    ):
        BACON_DIR = directory
        break

if BACON_DIR is None:
    raise FileNotFoundError(
        "BACion package directory could not be found."
    )

os.chdir(BACON_DIR)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-r",
    "requirements.txt"
])

subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-e",
    "."
], check=True)

BACON_SRC = BACON_DIR / "src"

if str(BACON_SRC) not in sys.path:
    sys.path.insert(0, str(BACON_SRC))

from bacion import BACionPredictor
from google.colab import files
from Bio import SeqIO

predictor = BACionPredictor(
    model_path=str(
        BACON_DIR / "model" / "bacion_xgboost.json"
    ),
    feature_config_path=str(
        BACON_DIR / "model" / "feature_config.json"
    ),
    feature_names_path=str(
        BACON_DIR / "model" / "feature_names.json"
    ),
)

uploaded = files.upload()

filename, file_bytes = next(iter(uploaded.items()))

input_path = Path("/content") / filename

with open(input_path, "wb") as f:
    f.write(file_bytes)

records = list(
    SeqIO.parse(str(input_path), "fasta")
)

if not records:
    raise ValueError(
        "No protein sequences were found in the uploaded FASTA file."
    )

result = predictor.predict(str(input_path))

predicted_nacl = float(
    result.get("predicted_maximum_NaCl_percent", 0.0)
)

print(
    f"Predicted maximum NaCl concentration supporting bacterial growth: "
    f"{predicted_nacl:.2f}% ± {FINAL_TEST_MAE:.2f}%"
)
